# 03 - Demand exploration and anomaly register

Profile SKU-region demand, zero-month frequency, intermittency class, family seasonality, lifecycle, customer/channel transitions, and high spikes. The anomaly register separates known supply events and likely source errors from unusual demand that still needs business review.

Rows outside Jan 2019-Oct 2026 are excluded from history profiles but remain in the register. A large observation is not automatically deleted: check the event calendar and customer history first.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "outputs" / "demand_reconstruction").is_dir():
            return candidate
    raise FileNotFoundError("Run notebook 02 first; reconstructed demand was not found.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src.demand_exploration import run_demand_exploration

summary = run_demand_exploration(PROJECT_ROOT)
for key, value in summary.items():
    print(f"{key}: {value}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

output_dir = PROJECT_ROOT / "outputs" / "demand_exploration"
profiles = pd.read_csv(output_dir / "demand_profile_by_series.csv", low_memory=False)
anomalies = pd.read_csv(output_dir / "anomaly_register.csv", low_memory=False)
seasonality = pd.read_csv(output_dir / "family_seasonality.csv")

print("Intermittency classes")
display(profiles["Intermittency_Class"].value_counts().rename_axis("Class").to_frame("Series"))
print("Anomaly types and classifications")
display(anomalies.groupby(["Anomaly_Type", "Classification"], dropna=False).size().rename("Rows").reset_index())
print("Example customer/channel transitions")
display(pd.read_csv(output_dir / "customer_channel_switches.csv").head(20))

plot_data = seasonality.pivot(index="Month_Number", columns="Product_Family", values="Mean_Monthly_Demand")
axis = plot_data.plot(figsize=(12, 5), marker="o")
plt.title("Average monthly demand by product family and calendar month")
plt.xlabel("Calendar month")
plt.ylabel("Average units")
plt.xticks(range(1, 13))
plt.grid(alpha=0.25)
plt.tight_layout()
seasonality_chart = output_dir / "family_seasonality.png"
axis.figure.savefig(seasonality_chart, dpi=150, bbox_inches="tight")
plt.close(axis.figure)
display(Image(filename=str(seasonality_chart)))

assert len(profiles) == summary["sku_region_series"]
assert anomalies["Anomaly_Type"].notna().all()
print("\nValidation passed: profiles and anomaly register were generated.")

Outputs are in `outputs/demand_exploration/`: series demand profiles, monthly family totals, family seasonality, customer/channel switches, and the anomaly register with suggested action and any matching event IDs.